# Classical Reasoning System: entailment over rules, causal graphs and logic stories

**Task.** Given a knowledge base (context) and a query (statement), decide whether the query is entailed: `True` / `False`.

**Pipeline.** text → router → translator → (facts, rules, interventions, goal) → **forward chaining** → True/False

One classical reasoning engine serves RuleTaker, CounterBench and FOLIO; only the translator in front of it differs.
The translators are regular-expression templates. No generative model is used, and the decision always comes from the engine.

**To generate `submission.csv`:** set `TEST_PATH` in the last cell and run it. The sections before it (data loading and experiments) are optional for that step.

## 1. Public data (development and experiments only)
If the datasets cannot be downloaded the notebook still works: experiment cells are skipped and the engine and submission cells run normally.

In [1]:
RUN_EXPERIMENTS = True
try:
    from datasets import load_dataset
    try:                                    # optional: only needed if Hugging Face asks for a token
        from huggingface_hub import notebook_login
        notebook_login()
    except Exception:
        pass
    ds_rule_taker = load_dataset("jise/ruletaker")
    ds_counter_bench = load_dataset("CounterBench/CounterBench", data_files="data_balanced_alpha_V1.json")
    ds_folio = load_dataset("yale-nlp/FOLIO")
    train_rule_taker = ds_rule_taker["train"]
    train_counter_bench = ds_counter_bench["train"]
    train_folio = ds_folio["train"]
except Exception as e:
    print("Public datasets not available, experiments will be skipped:", e)
    RUN_EXPERIMENTS = False

README.md:   0%|          | 0.00/955 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  242kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 66.5kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/ood_test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  113kB            

... (8 outputs left out)

folio_v2_train.jsonl:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

folio_v2_validation.jsonl:   0%|          | 0.00/208k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1001 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/203 [00:00<?, ? examples/s]

## 2. Reasoning engine: forward chaining to a fixpoint
* **atom**: a hashable name for a proposition; **literal** = (atom, polarity); **rule** = (list of body literals, head atom).
* **interventions** (`fixed`): atoms forced to True/False from outside, used for counterfactual questions.
* A negative literal holds when its atom is not derived (negation as failure).
* Each pass recomputes the derived set from the facts, so negation stays correct on acyclic graphs; for positive-only rules it is the usual least fixpoint.

In [2]:
import re, time, csv
from collections import Counter

# literal = (atom, positive?)   rule = ([literals], head_atom)
NEG_MODE = "naf"        # "naf" (current) | "strict"

def run_engine(facts, rules, fixed=None):
    fixed = fixed or {}
    forced_true = {a for a, v in fixed.items() if v}
    forced_false = {a for a, v in fixed.items() if not v}
    heads = {h for _, h in rules}
    base = (set(facts) - forced_false) | forced_true
    known, why = set(base), {}

    def holds(atom, positive):
        if positive:
            return atom in known
        if NEG_MODE == "strict":        # absence counts only if it is set from outside
            return atom in forced_false or (atom not in known and atom not in heads)
        return atom not in known

    for _ in range(len(rules) + 2):
        new = set(base)
        for body, head in rules:
            if head in fixed:
                continue
            if all(holds(a, p) for a, p in body):
                new.add(head)
                why.setdefault(head, body)
        if new == known:
            break
        known = new
    return known, why

def explain(atom, why, seen=None, depth=0):
    """Print the derivation chain for an atom (for the report's reasoning traces)."""
    seen = seen if seen is not None else set()
    if atom in seen or atom not in why:
        print("  " * depth + f"{atom}  [given]")
        return
    seen.add(atom)
    print("  " * depth + f"{atom}  <-  {why[atom]}")
    for a, positive in why[atom]:
        if positive:
            explain(a, why, seen, depth + 1)

## 3. Translators (natural language → facts and rules)
### 3.1 RuleTaker
Facts are the lines under `Facts:`. Each rule line (five templates) becomes one rule `A -> B`. Atoms are normalised sentences.

In [3]:
RT_PATTERNS = [
    re.compile(r"^assuming (.+?), we can say (.+)$"),
    re.compile(r"^if (.+?), then (.+)$"),
    re.compile(r"^based on (.+?), it is evident that (.+)$"),
    re.compile(r"^given that (.+?), it follows that (.+)$"),
    re.compile(r"^when (.+?), we can conclude (.+)$"),
]

def norm(text):
    text = text.lower().strip().rstrip(".")
    text = re.sub(r"\bthe\s+", "", text)
    return " ".join(text.split())

def frontend_ruletaker(context, statement):
    """RuleTaker text -> (facts, rules, no interventions, goal atom)."""
    facts, rules, section = set(), [], None
    for line in context.split("\n"):
        line = line.strip()
        if line == "":
            continue
        if line == "Facts:":
            section = "facts"; continue
        if line == "Rules:":
            section = "rules"; continue
        if section == "facts":
            facts.add(norm(line))
        elif section == "rules":
            low = line.lower().rstrip(".")
            for pattern in RT_PATTERNS:
                m = pattern.match(low)
                if m:
                    rules.append(([(norm(m.group(1)), True)], norm(m.group(2))))
                    break
    return facts, rules, {}, norm(statement)

### 3.2 CounterBench
`A causes B` → `A -> B`; `A and B together cause C` → one rule with two body literals; `A or B causes C` → two rules;
`A causes not B` → rule with a negated body. Variables that are never an effect are root facts unless observed false.
The question yields the target atom and the interventions (`not X` forces X to False). Nested questions apply both interventions (`NESTED_MODE = "both"`).

In [4]:
def _lit(text):
    t = re.sub(r"^and\s+", "", text.strip().strip("."))
    neg = t.lower().startswith("not ")
    return (t[4:].strip() if neg else t.strip()), (not neg)

def _body_dnf(text):
    text = re.sub(r"\btogether\b", "", text).strip()
    text = re.sub(r"^and\s+", "", text)
    return [[_lit(a) for a in d.split(" and ")] for d in text.split(" or ")]

def _negate_dnf(dnf):
    result = [[]]
    for conj in dnf:                                  # not(C1 or C2) = not C1 and not C2
        options = [[(a, not p)] for a, p in conj]     # not(a and b) = not a or not b
        result = [r + o for r in result for o in options]
    return result

NESTED_MODE = "both"        # "both" | "first_only" | "last_only"  (compared in section 5)

def frontend_counterbench(context, statement):
    """CounterBench text -> (root facts, causal rules, interventions, target atom)."""
    text, observed = context.strip(), {}
    m = re.search(r"We observed (.+)$", text, re.S)
    if m:
        for part in re.split(r",\s*|\s+and\s+", m.group(1).strip(" .")):
            if part.strip():
                name, val = _lit(part)
                observed[name] = val
        text = text[:m.start()]
    text = re.sub(r"^\s*We know that\s*", "", text).strip().rstrip(". ")

    segments = re.split(r"\bcauses?\b", text)
    bodies, spec = [_body_dnf(segments[0])], []
    for k in range(1, len(segments)):
        pieces = [p.strip() for p in segments[k].split(",") if p.strip()]
        last = (k == len(segments) - 1)
        eff_pieces = pieces if last else pieces[:-1]
        effects = []
        for p in eff_pieces:
            for q in re.sub(r"^and\s+", "", p).split(" and "):
                effects.append(_lit(q))
        spec.append((bodies[-1], effects))
        if not last:
            bodies.append(_body_dnf(pieces[-1]))

    rules, heads, used = [], set(), set()
    for dnf, effects in spec:
        for name, positive in effects:
            for conj in (dnf if positive else _negate_dnf(dnf)):   # "causes not B": B = not body
                rules.append((conj, name))
                heads.add(name)
                used.update(a for a, _ in conj)
    roots = (used - heads) | {n for n in observed if n not in heads}
    facts = {r for r in roots if observed.get(r, True)}           # roots occur unless observed false

    q = statement.strip()
    mt = re.search(r"[Ww]ould\s+(\w+)\s+occur", q)
    if not mt:
        return facts, rules, {}, None
    target = mt.group(1)
    cond = re.split(r"\binstead of\b", q[:mt.start()] + " " + q[mt.end():])[0]
    atoms = heads | used | {target}
    ordered = [(m2.group(2), m2.group(1) is None)
               for m2 in re.finditer(r"(not\s+)?\b([A-Z]\w*)\b", cond) if m2.group(2) in atoms]
    if re.search(r"further suppose|based on this assumption", q, re.I) and len(ordered) > 1:
        if NESTED_MODE == "first_only":
            ordered = ordered[:1]
        elif NESTED_MODE == "last_only":
            ordered = ordered[-1:]
    return facts, rules, dict(ordered), target

### 3.3 FOLIO (deliberately small)
Only simple `X is a Y` facts and `All A are B` rules are translated (grounded per named entity). Sentences with or / not / some / either and similar words are skipped, so the translation is sound but incomplete.

In [5]:
UNSAFE = (" or ", " not ", "n't", " no ", "some ", "either", "neither", " if ", " and ",
          "only", "at least", "except", "exactly", " than ")
ENT = r"[A-Z][\w'.-]*(?: [A-Z][\w'.-]*)*"

def _parse_fact(sentence):
    m = re.match(rf"^({ENT}) (?:is|was) (?:an? |the )?(.+?)\.?$", sentence.strip())
    if m and not any(u in " " + m.group(2).lower() + " " for u in UNSAFE):
        return (norm(m.group(2)), norm(m.group(1)))        # atom = (predicate, entity)
    return None

def frontend_folio(context, statement):
    """FOLIO text -> (facts, grounded rules, no interventions, goal atom)."""
    facts, universals, entities = set(), [], set()
    for s in re.split(r"(?<=[.!?])\s+|\n", context):
        s = s.strip()
        if not s or any(u in " " + s.lower() + " " for u in UNSAFE):
            continue                                       # skip what we cannot translate soundly
        m = re.match(r"^(?:all|every|each) (.+?) (?:is|are) (?:an? )?(.+?)\.?$", s, re.I)
        if m:
            universals.append((norm(m.group(1)), norm(m.group(2))))
            continue
        f = _parse_fact(s)
        if f:
            facts.add(f); entities.add(f[1])
    goal = _parse_fact(statement)
    if goal:
        entities.add(goal[1])
    rules = [([((a, e), True)], (b, e)) for a, b in universals for e in entities]
    return facts, rules, {}, goal

## 4. Router, solver and evaluation
The router chooses the translator from the text (`Facts:`/`Rules:` markers → RuleTaker; "cause" in the context and "occur" in the question → CounterBench; otherwise FOLIO).
`FOLIO_DEFAULT` is the answer for FOLIO when the conclusion cannot be derived (the label `Uncertain` is treated as not entailed).

In [6]:
FRONTENDS = {"ruletaker": frontend_ruletaker, "counterbench": frontend_counterbench, "folio": frontend_folio}

# ---- final settings ----
FOLIO_DEFAULT = False       # prediction for FOLIO when nothing is derived

def detect(context, statement=""):
    if "Facts:" in context and "Rules:" in context:
        return "ruletaker"
    if re.search(r"\bcauses?\b", context) and (statement == "" or re.search(r"\boccur", statement)):
        return "counterbench"
    return "folio"

def solve(context, statement):
    """Return True/False: is `statement` entailed by `context`?"""
    kind = detect(context, statement)
    derived = False
    try:
        facts, rules, fixed, goal = FRONTENDS[kind](context, statement)
        if goal is not None:
            known, _ = run_engine(facts, rules, fixed)
            derived = goal in known
    except Exception:
        derived = False                       # never crash on an unseen shape
    if kind == "folio" and not derived:
        return FOLIO_DEFAULT
    return derived

def rows_ruletaker(ds):    return [(r["context"], r["statement"], str(r["flag"]) == "True", r["depth"]) for r in ds]
def rows_counterbench(ds): return [(r["given_info"], r["question"], r["answer"].strip().lower() == "yes", r["type"]) for r in ds]
def rows_folio(ds):        return [(r["premises"], r["conclusion"], str(r["label"]) == "True", "all") for r in ds]

def evaluate(name, rows):
    start = time.time()
    ok, tot, wrong = Counter(), Counter(), []
    for i, (ctx, stmt, label, group) in enumerate(rows):
        good = (solve(ctx, stmt) == label)
        tot[group] += 1; ok[group] += good
        if not good:
            wrong.append(i)
    total_time = time.time() - start
    time_per_sample = total_time / len(rows)

    n_ok = sum(ok.values())
    print(f"{name}: {100 * n_ok / len(rows):.1f}% ({n_ok}/{len(rows)}) | "
          f"total {total_time:.2f}s | per sample {time_per_sample:.5f}s")
    if len(tot) > 1:
        for g in sorted(tot, key=str):
            print(f"    {g}: {100 * ok[g] / tot[g]:.1f}% ({ok[g]}/{tot[g]})")
    return wrong, time_per_sample

### Results on the public data
Expected output: RuleTaker 98.3%, CounterBench 96.0%, FOLIO 61.3% (total reasoning time well under one second per dataset).

In [20]:
if RUN_EXPERIMENTS:
    rt_rows = rows_ruletaker(train_rule_taker)
    cb_rows = rows_counterbench(train_counter_bench)
    fo_rows = rows_folio(train_folio)
    w_rt, t_rt = evaluate("RuleTaker", rt_rows)
    w_cb, t_cb = evaluate("CounterBench", cb_rows)
    w_fo, t_fo = evaluate("FOLIO", fo_rows)

    avg_time = (t_rt + t_cb + t_fo) / 3
    print(f"\nAverage time per sample over all three datasets: {avg_time:.5f} seconds")

RuleTaker: 98.3% (983/1000) | total 0.05s | per sample 0.00005s
    1: 100.0% (196/196)
    2: 96.8% (393/406)
    3: 99.0% (394/398)
CounterBench: 96.0% (960/1000) | total 0.07s | per sample 0.00007s
    basic: 100.0% (250/250)
    conditional: 92.0% (230/250)
    joint: 96.0% (240/250)
    nested: 96.0% (240/250)
FOLIO: 61.3% (614/1001) | total 0.05s | per sample 0.00005s

Average time per sample over all three datasets: 0.00006 seconds


## 5. Experiments
### 5.1 Reading of nested counterfactuals
Three readings compared on the 250 nested CounterBench questions: apply both interventions, only the first, only the last.

In [8]:
if RUN_EXPERIMENTS:
    nested = [r for r in cb_rows if r[3] == "nested"]
    for mode in ["both", "first_only", "last_only"]:
        NESTED_MODE = mode
        evaluate(f"nested / {mode}", nested)
    NESTED_MODE = "both"        # final choice (best of the three)

nested / both: 96.0% (240/250) | total 0.03s | per sample 0.00010s
nested / first_only: 76.0% (190/250) | total 0.03s | per sample 0.00010s
nested / last_only: 80.0% (200/250) | total 0.02s | per sample 0.00008s


### 5.2 Reading of negated causes
`naf`: a negated condition holds when its atom is not derived. `strict`: absence counts only if the atom was forced false or is a root. The strict variant was rejected.

In [9]:
if RUN_EXPERIMENTS:
    for mode in ["naf", "strict"]:
        NEG_MODE = mode
        evaluate(f"CounterBench / {mode}", cb_rows)
    NEG_MODE = "naf"            # final choice

CounterBench / naf: 96.0% (960/1000) | total 0.08s | per sample 0.00008s
    basic: 100.0% (250/250)
    conditional: 92.0% (230/250)
    joint: 96.0% (240/250)
    nested: 96.0% (240/250)
CounterBench / strict: 70.5% (705/1000) | total 0.07s | per sample 0.00007s
    basic: 62.0% (155/250)
    conditional: 74.0% (185/250)
    joint: 76.0% (190/250)
    nested: 70.0% (175/250)


### 5.3 Error analysis
Unparsed RuleTaker rule lines, two RuleTaker errors, three CounterBench conditional errors, and how often the FOLIO translator derives anything.

In [15]:
if RUN_EXPERIMENTS:
    # evaluate() returns (wrong_indices, time_per_sample); keep only the indices
    if isinstance(w_rt, tuple): w_rt = w_rt[0]
    if isinstance(w_cb, tuple): w_cb = w_cb[0]

    # --- RuleTaker: rule lines no template matched, plus two misses
    unparsed = Counter()
    for ctx, stmt, label, _ in rt_rows:
        sec = None
        for line in ctx.split("\n"):
            line = line.strip()
            if line in ("Facts:", "Rules:"):
                sec = line; continue
            if sec == "Rules:" and line and not any(p.match(line.lower().rstrip(".")) for p in RT_PATTERNS):
                unparsed[line] += 1
    print("unparsed rule lines:", len(unparsed), list(unparsed)[:5])

    for i in w_rt[:2]:
        ctx, stmt, label, depth = rt_rows[i]
        print(ctx)
        print("STATEMENT:", stmt, "| label:", label, "| ours:", solve(ctx, stmt), "\n")

    # --- CounterBench: three conditional misses with their parsed rules
    w_cond = [i for i in w_cb if cb_rows[i][3] == "conditional"]
    for i in w_cond[:3]:
        ctx, stmt, label, _ = cb_rows[i]
        facts, rules, fixed, goal = frontend_counterbench(ctx, stmt)
        print("GIVEN:", ctx)
        print("QUESTION:", stmt, "| label:", label, "| ours:", solve(ctx, stmt))
        print("facts:", facts, "| fixed:", fixed, "| goal:", goal)
        for r in rules:
            print("   ", r)
        print()

    # --- FOLIO: how often does the translator derive anything?
    n_goal = n_derived = n_correct = 0
    for ctx, stmt, label, _ in fo_rows:
        facts, rules, fixed, goal = frontend_folio(ctx, stmt)
        if goal is None:
            continue
        n_goal += 1
        known, _ = run_engine(facts, rules)
        if goal in known:
            n_derived += 1
            n_correct += label
    print(f"FOLIO: conclusion parsed {n_goal}/{len(fo_rows)} | derived True {n_derived} | of those correct {n_correct}")

unparsed rule lines: 0 []
Facts:
The train plays football.
The artist is fluffy.
The train plays instruments.
Rules:
Based on the train plays instruments, it is evident that the cat plays instruments.
Given that the train plays instruments, it follows that the mountain is fluffy.
Based on the artist is fluffy, it is evident that the cat plays instruments.
Given that the train plays football, it follows that the mountain plays football.
If the train plays instruments, then the artist is fluffy.
Assuming the train plays instruments, we can say the artist is fast.
STATEMENT: The artist is fast. | label: False | ours: True 

Facts:
The dog makes pizza.
The pig is an artist.
Emma makes pizza.
Rules:
Given that emma makes pizza, it follows that the pig cooks well.
When the dog makes pizza, we can conclude the pig is an artist.
Assuming emma cooks well, we can say the pig is an artist.
When emma makes pizza, we can conclude alice is an artist.
Based on emma cooks well, it is evident that alic

## 6. Reasoning traces
Derived atom, then the rule body that first produced it, then its supports.

In [14]:
def show_trace(context, statement):
    kind = detect(context, statement)
    facts, rules, fixed, goal = FRONTENDS[kind](context, statement)
    known, why = run_engine(facts, rules, fixed)
    print("kind:", kind, "| goal:", goal, "| derived:", goal in known, "| fixed:", fixed)
    if goal in known:
        explain(goal, why)

In [16]:
if RUN_EXPERIMENTS:
    # RuleTaker example
    rt_example = train_rule_taker[2]
    show_trace(rt_example["context"], rt_example["statement"])
    print()

    # CounterBench example (first conditional question with answer "yes")
    cb_example = None
    for row in train_counter_bench:
        if row["type"] == "conditional" and row["answer"] == "yes":
            cb_example = row
            break

    if cb_example is None:
        print("No conditional 'yes' example found in CounterBench.")
    else:
        print(cb_example["given_info"])
        print(cb_example["question"])
        show_trace(cb_example["given_info"], cb_example["question"])

kind: ruletaker | goal: mike is colorful | derived: True | fixed: {}
mike is colorful  <-  [('globe is colorful', True)]
  globe is colorful  [given]

We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune
Would Lumbo occur if not Ziklo instead of Ziklo?
kind: counterbench | goal: Lumbo | derived: True | fixed: {'Ziklo': False}
Lumbo  <-  [('Vork', True)]
  Vork  <-  [('Blaf', True), ('Trune', True)]
    Blaf  <-  [('Ziklo', False)]
    Trune  [given]


## 7. Generate `submission.csv`
Set `TEST_PATH` to the test file (`.csv`, `.tsv`, `.json`, `.jsonl` or `.parquet`) and run this cell. It finds the columns by name
(context: `context` / `given_info` / `premises`; statement: `statement` / `question` / `conclusion`; id: `id` / `question_id` / `example_id`, otherwise the row number),
writes `id,label`, and prints the number of rows, unreadable rows (should be 0) and the total inference time.

In [17]:
import os, time, csv
import pandas as pd

# ================== the only line to change when the test file arrives ==================
TEST_PATH = "test.csv"            # .csv / .tsv / .json / .jsonl / .parquet
OUT_PATH  = "submission.csv"
# =========================================================================================

# ---- final settings (reset anything the experiments changed) ----
NEG_MODE = "naf"
NESTED_MODE = "both"
FOLIO_DEFAULT = False             # prediction for FOLIO when nothing is derived

# ---- reading the test file ----
CTX_COLS  = ["context", "given_info", "premises", "knowledge_base", "kb", "text"]
STMT_COLS = ["statement", "question", "conclusion", "query", "hypothesis"]
ID_COLS   = ["id", "question_id", "example_id", "idx", "index"]

def _missing(v):
    return v is None or (isinstance(v, float) and v != v)

def _pick(row, names):
    for n in names:
        v = row.get(n)
        if not _missing(v):
            return v
    return None

def load_test(path):
    if path.endswith(".jsonl"):
        df = pd.read_json(path, lines=True)
    elif path.endswith(".json"):
        df = pd.read_json(path)
    elif path.endswith(".parquet"):
        df = pd.read_parquet(path)
    elif path.endswith(".tsv"):
        df = pd.read_csv(path, sep="\t")
    else:
        df = pd.read_csv(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df.to_dict("records")

def make_submission(rows, path=OUT_PATH):
    start, unreadable = time.time(), 0
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "label"])
        for i, row in enumerate(rows):
            ctx, stmt = _pick(row, CTX_COLS), _pick(row, STMT_COLS)
            rid = _pick(row, ID_COLS)
            rid = i if rid is None else rid
            if ctx is None or stmt is None:
                unreadable += 1
                pred = FOLIO_DEFAULT
            else:
                pred = solve(str(ctx), str(stmt))
            w.writerow([rid, pred])
    print(f"wrote {path} | rows: {len(rows)} | unreadable: {unreadable} | inference time: {time.time() - start:.2f}s")
    return unreadable

# ---- run ----
if not os.path.exists(TEST_PATH):
    print(f"Test file '{TEST_PATH}' not found. Set TEST_PATH above and run this cell again.")
else:
    test_rows = load_test(TEST_PATH)
    print(len(test_rows), "rows | columns:", list(test_rows[0].keys()))
    bad = make_submission(test_rows)
    if bad:
        print("Some rows had no recognised context/statement column. Columns found above;",
              "add the right names to CTX_COLS / STMT_COLS.")
    sub = pd.read_csv(OUT_PATH)
    print(sub["label"].value_counts().to_string())
    print("ids unique:", sub["id"].is_unique)

Test file 'test.csv' not found. Set TEST_PATH above and run this cell again.


In [18]:
import csv
import time

if RUN_EXPERIMENTS:
    # (name, rows) for each dataset; every row is (context, statement, label, group)
    all_datasets = [
        ("ruletaker", rt_rows),
        ("counterbench", cb_rows),
        ("folio", fo_rows),
    ]

    start = time.time()
    n_written = 0

    with open("submission.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])

        for name, rows in all_datasets:
            for i, row in enumerate(rows):
                context = row[0]
                statement = row[1]
                prediction = solve(context, statement)   # True or False

                row_id = name + "_" + str(i)             # e.g. ruletaker_0, folio_12
                writer.writerow([row_id, prediction])
                n_written += 1

    total_time = time.time() - start
    print("wrote submission.csv with", n_written, "rows")
    print(f"total time: {total_time:.2f}s | per sample: {total_time / n_written:.5f}s")
else:
    print("Public datasets are not loaded, so nothing was written.")

wrote submission.csv with 3001 rows
total time: 0.20s | per sample: 0.00007s
